In [ ]:
import torch
import torchvision
from torch import nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from pathlib import Path

In [ ]:
from pathlib import Path

data_path = Path("AI_mfg_data")

In [ ]:
gt_paths_list = list((data_path / "gt_labels").glob("*.mat"))
images_paths_list = list((data_path / "train_images").glob("*.png"))

len(images_paths_list), len(gt_paths_list)

In [ ]:
image_ids = sorted(set([int(id.name.split(".")[0].split("_")[-2]) for id in images_paths_list]))
image_ids[:13]

In [ ]:
gt_label_ids = sorted(set([int(id.name.split(".")[0].split("_")[-1]) for id in gt_paths_list]))
len(gt_label_ids)
(gt_label_ids)

In [ ]:
unavailable_label_indexes = [i for i in image_ids if i not in gt_label_ids]
len(unavailable_label_indexes)

In [ ]:
48000/2-17923

In [ ]:
gt_paths_dict = gt_path_dict = {
	Path(path).stem.split("_")[-1]: path
	for path in gt_paths_list
}
gt_paths_dict

In [ ]:
from pathlib import Path
from collections import defaultdict

image_dict = defaultdict(dict)

for path in images_paths_list:
	filename = path.stem
	
	parts = filename.split("_")
	
	sample_id = parts[2]
	image_num = parts[3]
	
	image_dict[sample_id][image_num] = path

image_dict, len(image_dict)

In [ ]:
import pandas as pd

df_all = pd.DataFrame({
  "ids": image_ids
})

df_all['label_path'] = df_all['ids'].astype(str).map(gt_path_dict)

for i in [1, 2]:
	df_all[f"image_{i}"] = df_all["ids"].astype(str).map(
		lambda x: image_dict.get(x, {}).get(str(i))
	)

df_all = df_all[['image_1', 'image_2', 'label_path']]

df_all.head(5)

In [ ]:
df_filtered = df_all[df_all['label_path'].notna()]
df_missing_labels = df_all[df_all['label_path'].isna()]

In [ ]:
df_filtered['label_path'].count(), df_missing_labels['label_path'].count()

In [ ]:
from scipy import io

sample_label_path = gt_paths_list[42]

mat_file = io.loadmat(sample_label_path)
print(mat_file.keys())

print(mat_file['Disp_field_1'].shape)

In [ ]:
type(mat_file['Disp_field_1'])

In [ ]:
import torch
from torch.utils.data import Dataset
import pandas as pd
from torchvision import transforms
from scipy import io
from PIL import Image

class DICDataset(Dataset):
	def __init__(
		self, 
		paths_df: pd.DataFrame, 
		transform: transforms=None
	):

		required_columns = {'image_1', 'image_2', 'label_path'}
		if not required_columns.issubset(paths_df):
			raise NameError("The DataFrame must have ''image_1', 'image_2', 'label_path' as column names.")
		self.paths_df = paths_df.reset_index(drop=True)
		self.transform = transform

	def __len__(self):
		return len(self.paths_df)

	def __getitem__(self, idx):
		row = self.paths_df.iloc[idx]
		image_1_path = row['image_1']
		image_2_path = row['image_2']
		label_path = row['label_path']

		tensor_tranform = transforms.ToTensor()

		image_1 = Image.open(image_1_path)
		image_2= Image.open(image_2_path)

		if self.transform:
			image_1 = self.transform(image_1)
			image_2 = self.transform(image_2)

		mat = io.loadmat(label_path)

		if "Disp_field_1" in mat:
			disp = mat["Disp_field_1"]
		elif "Disp_field_2" in mat:
			disp = mat["Disp_field_2"]

		disp_label = torch.from_numpy(disp.transpose(2, 0, 1)).float()

		return tensor_tranform(image_1), tensor_tranform(image_2), disp_label

In [ ]:
sample_ds = DICDataset(paths_df=df_filtered[:2])

In [ ]:
sample_ds[0][0].shape,sample_ds[0][1].shape,sample_ds[0][2].shape,

In [ ]:
import torchvision

resnet_wts = torchvision.models.ResNet34_Weights.DEFAULT

model = torchvision.models.resnet34(weights=resnet_wts)

model

In [ ]:
for child in model.children():
    print(child._get_name())

In [ ]:
feature_extractor = nn.Sequential(*list(model.children())[:-4])
feature_extractor[0]

In [ ]:
feature_extractor[0] = nn.Conv2d(
    in_channels=1,
    out_channels=64,
    kernel_size=(7, 7),
    stride=(2, 2),
    padding=(3, 3),
    bias=False
)

In [ ]:
feature_extractor

In [ ]:
from torchinfo import summary

summary(
	model=feature_extractor,
	input_size=(1, 1, 256, 256),
	col_names=["input_size", "output_size", "num_params", "trainable"],
  	row_settings=["var_names"]
)

In [ ]:
class FeatureExtractor(nn.Module):
    def __init__(self, base_backbone):
        super().__init__()
        self.base_backbone = base_backbone

    def forward(self, initial, final):
        feat_initial = self.base_backbone(initial)
        feat_final = self.base_backbone(final)

        return feat_initial, feat_final

In [ ]:
feature_extractor_baseline = FeatureExtractor(base_backbone=feature_extractor)

In [ ]:
summary(
	model=feature_extractor_baseline,
	input_size=[(1, 1, 256, 256), (1, 1, 256, 256)],
	col_names=["input_size", "output_size", "num_params", "trainable"],
  	row_settings=["var_names"]
)

In [ ]:
rnd_img_tensors = (
    torch.rand(size=(1, 1, 256, 256)),
    torch.rand(size=(1, 1, 256, 256))
)

rnd_img_tensors

In [ ]:
rnd_sample_op = feature_extractor_baseline(*rnd_img_tensors)
rnd_sample_op[0].shape, rnd_sample_op[1].shape

In [ ]:
class SpatialTransformerEncoder(nn.Module):
    def __init__(
        self,
        embed_dim=128,
        num_heads=4,
        num_layers=2,
        ff_dim=256,
        dropout=0.1
    ):
        super().__init__()

        self.pos_embed = nn.Parameter(
            torch.randn(1, 32 * 32, embed_dim) * 0.02
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, features):
        tokens = features.flatten(2)

        tokens = tokens.transpose(1, 2)

        tokens = tokens + self.pos_embed

        tokens = self.encoder(tokens)
        tokens = self.norm(tokens)

        B, N, C = tokens.shape
        features = tokens.transpose(1, 2).reshape(B, C, 32, 32)

        return features

In [ ]:
encoder = SpatialTransformerEncoder()

In [ ]:
summary(
	model=encoder,
	input_size=[1, 128, 32, 32],
	col_names=["input_size", "output_size", "num_params", "trainable"],
  	row_settings=["var_names"]
)

In [ ]:
encoded_initial = encoder(rnd_sample_op[0])
encoded_final = encoder(rnd_sample_op[1])

encoded_initial.shape, encoded_final.shape

In [ ]:
encoded_initial = encoded_initial.flatten(2).transpose(1, 2)
encoded_final = encoded_final.flatten(2).transpose(1, 2)

encoded_initial.shape, encoded_final.shape

In [ ]:
class CrossAttention(nn.Module):
    def __init__(
        self,
        embed_dim=128,
        num_heads=4,
        dropout=0.1
    ):
        super().__init__()

        self.cross_attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm1 = nn.LayerNorm(embed_dim)

        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 4, embed_dim),
            nn.Dropout(dropout)
        )

        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, query_features, context_features):
        # Query: initial image
        # Key/value: final image

        attended, _ = self.cross_attn(
            query=query_features,
            key=context_features,
            value=context_features,
            need_weights=False
        )

        x = self.norm1(query_features + attended)
        x = self.norm2(x + self.ffn(x))

        return x

In [ ]:
cross_attn_block = CrossAttention()

In [ ]:
cross_features = cross_attn_block(
    encoded_initial,
    encoded_final
)

cross_features.shape

In [ ]:
import torch.nn.functional as F


class GlobalMatching(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, feat_initial, feat_final):
        """
        Inputs:
            feat_initial: [B, 1024, 128]
            feat_final:   [B, 1024, 128]

        Output:
            flow: [B, 2, 32, 32]
            probabilities: [B, 1024, 1024]
        """

        B, N, C = feat_initial.shape

        feat_initial = F.normalize(feat_initial, p=2, dim=-1)
        feat_final = F.normalize(feat_final, p=2, dim=-1)

        similarity = torch.bmm(
            feat_initial,
            feat_final.transpose(1, 2)
        )

        probabilities = F.softmax(
            similarity / self.temperature,
            dim=-1
        )

        y, x = torch.meshgrid(
            torch.arange(32, device=feat_initial.device),
            torch.arange(32, device=feat_initial.device),
            indexing="ij"
        )

        coords = torch.stack(
            [x.reshape(-1), y.reshape(-1)],
            dim=-1
        ).to(dtype=feat_initial.dtype)

        matched_coords = torch.bmm(
            probabilities,
            coords.unsqueeze(0).expand(B, -1, -1)
        )

        reference_coords = coords.unsqueeze(0).expand(B, -1, -1)

        flow = matched_coords - reference_coords

        flow = flow.transpose(1, 2).reshape(B, 2, 32, 32)

        return flow, probabilities

In [ ]:
encoded_initial = encoder(rnd_sample_op[0])
encoded_final = encoder(rnd_sample_op[1])

tokens_initial = encoded_initial.flatten(2).transpose(1, 2)
tokens_final = encoded_final.flatten(2).transpose(1, 2)

matcher = GlobalMatching()

flow, probabilities = matcher(
    tokens_initial,
    tokens_final
)

print("Flow:", flow.shape)
print("Matching probabilities:", probabilities.shape)

In [ ]:
class DisplacementDecoder(nn.Module):
    def __init__(self, hidden_channels=64):
        super().__init__()

        self.refine_64 = nn.Sequential(
            nn.Conv2d(2, hidden_channels, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(hidden_channels, hidden_channels, 3, padding=1),
            nn.GELU(),
        )

        self.refine_128 = nn.Sequential(
            nn.Conv2d(hidden_channels, hidden_channels // 2, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(hidden_channels // 2, hidden_channels // 2, 3, padding=1),
            nn.GELU(),
        )

        self.refine_256 = nn.Sequential(
            nn.Conv2d(hidden_channels // 2, hidden_channels // 4, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(hidden_channels // 4, hidden_channels // 4, 3, padding=1),
            nn.GELU(),
        )

        self.predict_flow = nn.Conv2d(
            hidden_channels // 4, 2, kernel_size=3, padding=1
        )

    def forward(self, flow):
        # flow: [B, 2, 32, 32]

        # 32 -> 64
        x = F.interpolate(
            flow, scale_factor=2,
            mode="bilinear", align_corners=False
        )
        x = self.refine_64(x)

        # 64 -> 128
        x = F.interpolate(
            x, scale_factor=2,
            mode="bilinear", align_corners=False
        )
        x = self.refine_128(x)

        # 128 -> 256
        x = F.interpolate(
            x, scale_factor=2,
            mode="bilinear", align_corners=False
        )
        x = self.refine_256(x)

        # Predict a correction to the upsampled coarse flow
        correction = self.predict_flow(x)

        # Return coarse flow plus learned correction
        base_flow = F.interpolate(
            flow, size=(256, 256),
            mode="bilinear", align_corners=False
        )

        return base_flow + correction

In [ ]:
decoder = DisplacementDecoder()

coarse_flow = torch.randn(1, 2, 32, 32)
final_flow = decoder(coarse_flow)

print(final_flow.shape)

In [ ]:
class CharbonnierEndpointLoss(nn.Module):
    def __init__(self, eps=1e-3):
        super().__init__()
        self.eps = eps

    def forward(self, pred, target):
        # pred, target: [B, 2, H, W]

        error = pred - target

        # Combine horizontal and vertical displacement errors
        endpoint_error_sq = torch.sum(error ** 2, dim=1)

        # Robust, differentiable endpoint loss
        loss = torch.sqrt(endpoint_error_sq + self.eps ** 2)

        return loss.mean()

In [ ]:
from models.crossflow import TransformerDIC

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = TransformerDIC(pretrained=False).to(device)

initial = torch.randn(2, 1, 256, 256, device=device)
final = torch.randn(2, 1, 256, 256, device=device)

with torch.no_grad():
    prediction = model(initial, final)

print("Initial image:", initial.shape)
print("Final image:  ", final.shape)
print("Prediction:   ", prediction.shape)

In [ ]:
from torch.utils.data import DataLoader

SEED = 42
DATA_FRAC = 0.3
TEST_SIZE = 0.2
BATCH_SIZE = 32

dataset_paths = df_filtered.sample(frac=DATA_FRAC, random_state=SEED)

test_paths_df = dataset_paths.sample(frac=TEST_SIZE, random_state=SEED)
train_paths_df = dataset_paths.drop(test_paths_df.index)

train_dataset = DICDataset(paths_df=train_paths_df)
test_dataset = DICDataset(paths_df=test_paths_df)

print(f"[INFO] Training on {int(DATA_FRAC * 100)}% of samples.")
print(f"[INFO] Total train samples: {len(train_dataset)}.")
print(f"[INFO] Total test samples: {len(test_dataset)}.")

train_dl = DataLoader(
    dataset=train_dataset,
    shuffle=True,
    batch_size=BATCH_SIZE
)

test_dl = DataLoader(
    dataset=test_dataset,
    shuffle=False,
    batch_size=BATCH_SIZE
)

print(f"[INFO] Train dataloader size: {len(train_dl)}")
print(f"[INFO] Test dataloader size: {len(test_dl)}")

In [ ]:
from models.crossflow import TransformerDIC, CharbonnierEndpointLoss
from scripts.engine import train
import torch

device = "mps"

model = TransformerDIC()
loss_func = CharbonnierEndpointLoss()
optimizer = torch.optim.Adam(
    lr=1e-3,
    params=model.parameters()
)
EPOCHS = 10

results = train(
    model=model,
    train_dataloader=train_dl,
    test_dataloader=test_dl,
    optimizer=optimizer,
    loss_fn=loss_func,
    epochs=EPOCHS,
    device=device
)